In [1]:
from torchvision import transforms
from torchvision.datasets import FashionMNIST

custom_transformations = transforms.Compose([
    transforms.Resize(size=[256, 256]), # resized pixel calculation -> InterpolationMode.BILINEAR
    transforms.CenterCrop(size=[224, 224]),
    transforms.Grayscale(num_output_channels=3), # convert 1-channel to 3-channel
    transforms.ToTensor(), # converts from PIL images to tensors and rescales the pixel values from 0-255 to 0-1
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # vgg16 configuration -> channel wise mean and std
])

train_dataset = FashionMNIST(root="./data/train", train=True, download=True, transform=custom_transformations)
test_dataset = FashionMNIST(root="./data/test", train=False, download=True, transform=custom_transformations)

print(train_dataset, test_dataset, sep="\n\n", end="\n\n")
print(f"shape of images → {train_dataset[0][0].shape}")
print(f"class labels → {train_dataset.classes}")

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: ./data/train
    Split: Train
    StandardTransform
Transform: Compose(
               Resize(size=[256, 256], interpolation=bilinear, max_size=None, antialias=True)
               CenterCrop(size=[224, 224])
               Grayscale(num_output_channels=3)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

Dataset FashionMNIST
    Number of datapoints: 10000
    Root location: ./data/test
    Split: Test
    StandardTransform
Transform: Compose(
               Resize(size=[256, 256], interpolation=bilinear, max_size=None, antialias=True)
               CenterCrop(size=[224, 224])
               Grayscale(num_output_channels=3)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

shape of images → torch.Size([3, 224, 224])
class labels → ['T-shirt/top', 'Trouser', 'Pullover', 'Dress

In [2]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
torch.cuda.get_device_name()

Using device: cuda


'NVIDIA GeForce MX350'

In [3]:
# custom dataset class
from torch.utils.data import Dataset

class FashionMNIST(Dataset):
    def __init__(self, data, transforms=None):
        self.data = data
        # transformations are recommended to apply in this block

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index][0], self.data[index][1]

In [4]:
# dataset object and dataloaders
from torch.utils.data import DataLoader

train_df = FashionMNIST(data=train_dataset)
test_df = FashionMNIST(data=test_dataset)

train_loader = DataLoader(
    dataset=train_df,
    batch_size=32,
    shuffle=True,
    drop_last=False
)

test_loader = DataLoader(
    dataset=test_df,
    batch_size=32,
    shuffle=False,
    drop_last=False
)

In [5]:
import torchvision.models as models
from torchinfo import summary

# https://docs.pytorch.org/vision/main/models/generated/torchvision.models.vgg16.html
vgg16 = models.vgg16(weights="IMAGENET1K_V1") # path → /home/shriram/.cache/torch/hub/checkpoints/vgg16-397923af.pth
summary(vgg16, input_size=(32, 3, 224, 224))

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /home/shriram/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [02:41<00:00, 3.43MB/s] 


Layer (type:depth-idx)                   Output Shape              Param #
VGG                                      [32, 1000]                --
├─Sequential: 1-1                        [32, 512, 7, 7]           --
│    └─Conv2d: 2-1                       [32, 64, 224, 224]        1,792
│    └─ReLU: 2-2                         [32, 64, 224, 224]        --
│    └─Conv2d: 2-3                       [32, 64, 224, 224]        36,928
│    └─ReLU: 2-4                         [32, 64, 224, 224]        --
│    └─MaxPool2d: 2-5                    [32, 64, 112, 112]        --
│    └─Conv2d: 2-6                       [32, 128, 112, 112]       73,856
│    └─ReLU: 2-7                         [32, 128, 112, 112]       --
│    └─Conv2d: 2-8                       [32, 128, 112, 112]       147,584
│    └─ReLU: 2-9                         [32, 128, 112, 112]       --
│    └─MaxPool2d: 2-10                   [32, 128, 56, 56]         --
│    └─Conv2d: 2-11                      [32, 256, 56, 56]         29

In [6]:
vgg16.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [7]:
vgg16.features

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (17): Conv2d(256, 512, kernel_si

In [8]:
from torch import nn

for param in vgg16.features.parameters():
    param.requires_grad=False

vgg16.classifier = nn.Sequential(
    nn.Linear(25088, 1024),
    nn.ReLU(),
    nn.Dropout(0.5),

    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.5),

    nn.Linear(512, 10)
).to(device) # convolution and pooling base remains freezed, only a neural network and classification head is trainable

summary(vgg16, input_size=(32, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
VGG                                      [32, 10]                  --
├─Sequential: 1-1                        [32, 512, 7, 7]           --
│    └─Conv2d: 2-1                       [32, 64, 224, 224]        (1,792)
│    └─ReLU: 2-2                         [32, 64, 224, 224]        --
│    └─Conv2d: 2-3                       [32, 64, 224, 224]        (36,928)
│    └─ReLU: 2-4                         [32, 64, 224, 224]        --
│    └─MaxPool2d: 2-5                    [32, 64, 112, 112]        --
│    └─Conv2d: 2-6                       [32, 128, 112, 112]       (73,856)
│    └─ReLU: 2-7                         [32, 128, 112, 112]       --
│    └─Conv2d: 2-8                       [32, 128, 112, 112]       (147,584)
│    └─ReLU: 2-9                         [32, 128, 112, 112]       --
│    └─MaxPool2d: 2-10                   [32, 128, 56, 56]         --
│    └─Conv2d: 2-11                      [32, 256, 56, 56]   

In [9]:
learning_rate = 0.001
epochs = 5

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=vgg16.classifier.parameters(), lr=learning_rate) # only pass those parameters who gets trained
vgg16.train()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
for epoch in range(epochs):
    total_epoch_loss = 0

    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = vgg16(batch_features)

        loss = criterion(outputs, batch_labels)
        loss.backward()

        optimizer.zero_grad()
        optimizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()

    avg_loss = total_epoch_loss / len(train_loader)
    print(f"Epoch: {epoch + 1} - Loss: {avg_loss}")

In [ ]:
vgg16.eval()
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = vgg16(batch_features)
        _, predicted = torch.max(outputs, 1)
        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)